# 


https://pydantic.dev/docs/ai/overview/

https://pydantic.dev/docs/ai/examples/getting-started/weather-agent/




## 

# 세팅  

In [ ]:
! uv add deepagents

In [ ]:
! uv add -U langchain langchain-core langchain-google-genai deepagents langgraph

In [3]:
! uv add -U "deepagents[quickjs]"

Resolved 286 packages in 3.05s
 Downloaded hf-xet
 Downloaded wasmtime
 Downloaded ruff
 Downloaded openai
 Downloaded litellm
Prepared 23 packages in 11.75s
Uninstalled 19 packages in 3.02s
Installed 23 packages in 4.73s
 + bsdiff4==1.2.6
 - cbor2==6.1.3
 + cbor2==6.1.4
 - cffi==2.1.0
 + cffi==2.1.1
 - deepagents==0.6.12
 + deepagents==0.7.3
 - fastapi==0.140.13
 + fastapi==0.141.1
 - feedparser==6.0.13
 + feedparser==6.0.14
 - feedparser-sgmllib==1.0.0
 + feedparser-sgmllib==2.1.0
 - filelock==3.32.0
 + filelock==3.32.2
 - h2==4.4.0
 + h2==4.4.1
 - hf-xet==1.5.2
 + hf-xet==1.6.0
 - huggingface-hub==1.25.1
 + huggingface-hub==1.26.0
 - ipython==9.15.0
 + ipython==9.16.1
 - langchain-core==1.5.2
 + langchain-core==1.5.3
 + langchain-quickjs==0.3.5
 - langsmith==0.10.11
 + langsmith==0.10.15
 - litellm==1.94.0
 + litellm==1.95.0
 - openai==2.50.0
 + openai==2.53.0
 + quickjs-rs==0.2.5
 - ruff==0.16.0
 + ruff==0.16.1
 - traitlets==5.15.1
 + traitlets==5.16.1
 - typer==0.27.0
 + typer==0.

In [3]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

# Lab 1: 




In [8]:
from typing import Literal  # 값을 정해진 문자열 중 하나로 제한하는 타입

from pydantic import BaseModel, Field  # 구조화 출력 스키마 정의와 필드 제약 조건

from pydantic_ai import Agent, RunContext  # 에이전트 본체와 도구에 전달되는 실행 컨텍스트


class Sentiment(BaseModel):
    """에이전트의 출력 스키마. 모델은 반드시 이 구조에 맞는 값을 생성한다"""
    label: Literal['positive', 'negative', 'neutral']  # 감성 분류 결과. 세 값 중 하나만 허용
    score: float = Field(ge=-1, le=1)  # 감성 점수. ge/le로 -1 이상 1 이하로 범위 제한


# 에이전트 생성. output_type을 지정하면 자유 텍스트 대신 Sentiment 객체가 반환된다
agent = Agent('openai:gpt-5-mini', output_type=Sentiment)


@agent.tool  # 이 함수를 에이전트가 호출할 수 있는 도구로 등록
def recent_reviews(ctx: RunContext[None], product: str) -> list[str]:
    """제품에 대한 최근 리뷰 조각들을 가져옵니다."""
    # ctx는 실행 컨텍스트(의존성 타입이 None), product는 모델이 채워 넣는 인자
    return ['새 릴리스가 제가 불평했던 걸 전부 고쳐놨어요!']  # 실제 조회 대신 고정값을 반환하는 예시


# 에이전트 실행. 노트북에서는 run_sync 대신 await를 사용한다
result = await agent.run('Extract 앱에 대해 사람들이 어떻게 느끼고 있나요?')
print(result.output)  # Sentiment 인스턴스가 출력된다
#> label='positive' score=0.9

label='positive' score=0.9


# Lab 2: Weather Agent




In [12]:
from __future__ import annotations as _annotations  # 타입 힌트를 문자열로 지연 평가

import asyncio  # 동시 요청(gather)용
from dataclasses import dataclass  # 의존성 컨테이너를 간단히 정의
from typing import Any  # 반환 딕셔너리의 값 타입이 자유로울 때 사용

from httpx import AsyncClient  # 비동기 HTTP 클라이언트
from pydantic import BaseModel  # 응답 JSON 검증용 스키마
from pydantic_ai import Agent, RunContext  # 에이전트 본체와 도구에 전달되는 실행 컨텍스트


@dataclass
class Deps:
    """도구들이 공유하는 의존성 묶음"""
    client: AsyncClient  # 모든 도구가 함께 쓰는 HTTP 클라이언트


weather_agent = Agent(
    'openai:gpt-5-mini',
    # 'Be concise, reply with one sentence.' 정도면 openai 같은 일부 모델은 아래 도구들을
    # 적절히 사용하지만, anthropic이나 gemini 같은 모델은 좀 더 상세한 지시가 필요하다.
    instructions='간결하게, 한 문장으로 답하세요.',
    deps_type=Deps,  # 도구의 ctx.deps에 들어올 타입을 선언
    retries=2,  # 도구 호출이나 출력 검증 실패 시 최대 재시도 횟수
)


class LatLng(BaseModel):
    """위경도 응답을 검증할 스키마"""
    lat: float  # 위도
    lng: float  # 경도


@weather_agent.tool  # 이 함수를 에이전트가 호출할 수 있는 도구로 등록
async def get_lat_lng(ctx: RunContext[Deps], location_description: str) -> LatLng:
    """어떤 위치의 위도와 경도를 가져옵니다.

    Args:
        ctx: 실행 컨텍스트입니다.
        location_description: 위치에 대한 설명입니다.
    """
    # 참고: 여기서 오는 응답은 무작위이며, 위치 설명과는 관련이 없다.
    r = await ctx.deps.client.get(
        'https://demo-endpoints.pydantic.workers.dev/latlng',
        params={'location': location_description},  # 쿼리 파라미터로 위치 설명 전달
    )
    r.raise_for_status()  # 4xx/5xx 응답이면 예외 발생
    return LatLng.model_validate_json(r.content)  # 응답 JSON을 LatLng 스키마로 검증하며 파싱


@weather_agent.tool
async def get_weather(ctx: RunContext[Deps], lat: float, lng: float) -> dict[str, Any]:
    """어떤 위치의 날씨를 가져옵니다.

    Args:
        ctx: 실행 컨텍스트입니다.
        lat: 위치의 위도입니다.
        lng: 위치의 경도입니다.
    """
    # 참고: 여기서 오는 응답들은 무작위이며, lat과 lng와는 관련이 없다.
    temp_response, descr_response = await asyncio.gather(  # 두 요청을 동시에 보내고 함께 대기
        ctx.deps.client.get(
            'https://demo-endpoints.pydantic.workers.dev/number',
            params={'min': 10, 'max': 30},  # 10~30 범위의 임의 숫자를 기온 대신 사용
        ),
        ctx.deps.client.get(
            'https://demo-endpoints.pydantic.workers.dev/weather',
            params={'lat': lat, 'lng': lng},
        ),
    )
    temp_response.raise_for_status()  # 기온 응답 상태 확인
    descr_response.raise_for_status()  # 날씨 설명 응답 상태 확인
    return {
        'temperature': f'{temp_response.text} °C',  # 응답 본문에 단위를 붙여 문자열로 구성
        'description': descr_response.text,
    }


# 노트북은 이미 이벤트 루프 위에서 동작하므로 asyncio.run()을 쓸 수 없다.
# main() 함수와 __main__ 가드를 없애고, 셀에서 최상위 await로 직접 실행한다.
async with AsyncClient() as client:  # 블록을 벗어나면 클라이언트 연결이 정리된다
    deps = Deps(client=client)  # 도구들이 쓸 의존성 구성
    result = await weather_agent.run(
        '런던과 윌트셔의 날씨는 어떤가요?', deps=deps  # 사용자 프롬프트와 의존성을 함께 전달
    )

print('응답:', result.output)  # 에이전트의 최종 응답 텍스트 출력

c:\Users\jino7\Desktop\2026_Ai_Python-Source\Pydantic_AI_26_8_27\.venv\Lib\site-packages\pydantic\json_schema.py:342: RuntimeWarning: coroutine 'main' was never awaited
  mapping[key] = getattr(self, method_name)


응답: 런던은 폭풍우에 기온 약 26°C이고, 윌트셔는 비가 오며 기온 약 12°C입니다.


# end




